## Utilidades

In [282]:
import pandas as pd
import numpy as np


In [283]:
contracts = pd.read_csv("contracts.csv")
customers = pd.read_csv("customers.csv")
usage     = pd.read_csv("usage.csv")

# 1. Descubrimiento y Comprensión de Datos

In [284]:
contracts.head(-10)
#customers.head(-10) 
#usage.head()

,con_id,cli_ref,p_type,val_mo,st,dt_start
0,CON_000001,C_00995,F1,51.48,0,2020-07-26
1,CON_000002,C_00098,M2,112.79,1,2021-11-26
2,CON_000003,C_00506,F1,30.34,1,2020-02-23
3,CON_000004,C_01821,F1,81.40,1,2022-07-18
4,CON_000005,C_01071,F1,77.26,1,2022-11-25
...,...,...,...,...,...,...
5485,CON_005486,C_00458,F1,22.28,1,2021-05-24
5486,CON_005487,C_04582,M1,93.24,1,2021-07-20
5487,CON_005488,C_00428,F2,50.49,1,2022-07-02
5488,CON_005489,C_02416,M2,51.89,1,2020-04-07


### Renombrar columnas

In [285]:
contracts.rename(columns={
    "con_id":   "contract_id", # PK unica = 5500 registros
    "cli_ref":  "customer_id", # FK a customers (3395 coincidencias, 2 se repiten)
    "p_type":   "plan_type", # tipo de plan 
    "val_mo":   "monthly_value", # valor mensual del contrato
    "st":       "status", # estado del contrato (3 valores: )
    "dt_start": "start_date", # fecha de inicio del contrato
}, inplace=True)

customers.rename(columns={
    "id_cli":       "customer_id", # PK unica (revisar, no es unica) = 4994 registros // solución: coincidir entre customer_id y name
    "full_name":    "name", # nombre del cliente
    "contact_info": "email", # Contacto
    "reg_date":     "registration_date", # fecha de registro
    "cat":          "category", # 4 valores : A, B, C, X
    "int_score":    "internal_score", # puntaje interno del cliente (0-100)
}, inplace=True)

usage.rename(columns={
    "u_id":  "usage_id", # PK unica
    "c_ref": "contract_id", # FK contracts
    "qty":   "quantity", # consumo
    "ts":    "timestamp", # fecha y hora del consumo
    "loc":   "location", # ubicacion del consumo
}, inplace=True)

### Exploración

#### Info tablas

In [286]:
customers.info() 
contracts.info()
usage.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        5000 non-null   str    
 1   name               5000 non-null   str    
 2   email              5000 non-null   str    
 3   registration_date  5000 non-null   str    
 4   category           5000 non-null   str    
 5   internal_score     5000 non-null   float64
dtypes: float64(1), str(5)
memory usage: 234.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 5500 entries, 0 to 5499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   contract_id    5500 non-null   str    
 1   customer_id    5500 non-null   str    
 2   plan_type      5500 non-null   str    
 3   monthly_value  5500 non-null   float64
 4   status         5500 non-null   int64  
 5   start_date     5500 non-null   str    
dtypes: float64(1), int6

In [287]:
customer_id_nunique = customers['customer_id'].nunique()
contracts_id_nunique = contracts['contract_id'].nunique()
customer_id_contract_nunique = contracts['customer_id'].nunique()
customer_id_nunique, contracts_id_nunique, customer_id_contract_nunique

(4994, 5500, 3395)

In [288]:
# Cuenta cuántos registros de contracts tienen un customer_id presente en customers y viseversa
coincidencias_C_in_Q = contracts['customer_id'].isin(customers['customer_id']).sum() 
coincidencias_Q_in_C = customers['customer_id'].isin(contracts['customer_id']).sum()
coincidencias_C_in_Q, coincidencias_Q_in_C

(np.int64(5497), np.int64(3399))

In [289]:
# Valores únicos comunes
comunes = len(set(contracts['customer_id']) & set(customers['customer_id']))
comunes  # 3393 coincidencias y 2 se repiten esto es porque un cliente puede 
         #tener más de un contrato, pero un contrato no puede tener más de un cliente. (1:N)

3393

#### Entendiendo inconsistencia customer_id

In [290]:
filas_duplicadas = customers[customers['customer_id'].duplicated(keep=False)].sort_values(by='registration_date')
filas_duplicadas 

,customer_id,name,email,registration_date,category,internal_score
4994,C_00001,User_4995,user_4995@domain.com,2019-07-13,A,35.17
4993,C_00001,User_4994,user_4994@domain.com,2020-05-26,B,49.27
4995,C_00001,User_4996,user_4996@domain.com,2021-08-04,C,83.61
4992,C_00001,User_4993,user_4993@domain.com,2022-01-31,A,59.40
4990,C_00001,User_4991,user_4991@domain.com,2022-02-12,X,50.34
0,C_00001,User_1,user_1@domain.com,2022-08-02,A,41.05
4991,C_00001,User_4992,user_4992@domain.com,2023-03-13,A,35.20


In [291]:
contract_C_01 = contracts[contracts['customer_id'] == 'C_00001']
contract_C_01

,contract_id,customer_id,plan_type,monthly_value,status,start_date
1838,CON_001839,C_00001,M2,112.97,-1,2020-11-17


In [292]:
contract_U_01 = usage[usage['contract_id'] == 'CON_001839']
contract_U_01

,usage_id,contract_id,quantity,timestamp,location
8327,U_0008328,CON_001839,15.72,2024-03-12 15:56:16,US
9048,U_0009049,CON_001839,1.29,2024-06-08 04:04:20,UK
10047,U_0010048,CON_001839,0.42,2024-10-20 13:09:40,US
18951,U_0018952,CON_001839,34.59,2024-03-06 22:00:13,ES
20086,U_0020087,CON_001839,2.83,2024-07-05 23:20:39,??
30360,U_0030361,CON_001839,11.77,2024-01-20 18:48:47,DE
31936,U_0031937,CON_001839,1.71,2024-12-02 02:16:53,ES
33495,U_0033496,CON_001839,8.97,2024-10-13 01:13:04,??
42835,U_0042836,CON_001839,2.00,2024-11-25 20:34:00,DE
46541,U_0046542,CON_001839,14.93,2024-04-17 03:23:21,FR


In [293]:
# Teoria: Si el formato de customer_id es consistente, debería ser algo como 'C_00001', 'C_00002', etc. Esto implica que el número después de 'C_' 
# debería coincidir entre contracts y customers para el mismo cliente.
# Uso .extract para string con regex hecha con IA (busca 1 o más dígitos) y luego convierto a int para comparar numéricamente.
id_num = customers['customer_id'].str.extract(r'(\d+)').astype(int)
name_num = customers['name'].str.extract(r'(\d+)').astype(int)

# Filtrar las filas donde NO coinciden (patrón se rompe)
no_coinciden = customers[id_num[0] != name_num[0]]
no_coinciden

# Entiendo que cada customer_id tiene coincidencia por name y en este caso de trata de 6 clientes que tenian mal asignado el custormer_id ("conclusión")

,customer_id,name,email,registration_date,category,internal_score
4990,C_00001,User_4991,user_4991@domain.com,2022-02-12,X,50.34
4991,C_00001,User_4992,user_4992@domain.com,2023-03-13,A,35.20
4992,C_00001,User_4993,user_4993@domain.com,2022-01-31,A,59.40
4993,C_00001,User_4994,user_4994@domain.com,2020-05-26,B,49.27
4994,C_00001,User_4995,user_4995@domain.com,2019-07-13,A,35.17
4995,C_00001,User_4996,user_4996@domain.com,2021-08-04,C,83.61


#### Entendiendo inconsistencia Email

In [294]:
# Extraer el número de la columna 'name' (ej: '4987')
num_name = customers['name'].str.extract(r'(\d+)')[0]

# Construir el email que debería tener según el patrón
email_esperado = "user_" + num_name + "@domain.com"

# Identificar las filas donde el email real no coincide
errores = customers['email'] != email_esperado

# Filtrar y contar
emails_incorrectos = customers[errores]
cantidad_errores = len(emails_incorrectos)

cantidad_errores, emails_incorrectos[['name', 'email', 'customer_id']]

(500,
            name         email customer_id
 9       User_10    invalid_10     C_00010
 19      User_20    invalid_20     C_00020
 29      User_30    invalid_30     C_00030
 39      User_40    invalid_40     C_00040
 49      User_50    invalid_50     C_00050
 ...         ...           ...         ...
 4959  User_4960  invalid_4960     C_04960
 4969  User_4970  invalid_4970     C_04970
 4979  User_4980  invalid_4980     C_04980
 4989  User_4990  invalid_4990     C_04990
 4999  User_5000  invalid_5000     C_05000
 
 [500 rows x 3 columns])

In [295]:
# y cuantos coinciden con el patrón invalid?

email_invalid = "invalid_" + num_name
errores_invalid = customers['email'] != email_invalid

cantidad_invalid = len(errores_invalid)
cantidad_invalid

5000

In [296]:
# Comprobar que no hay inconsistencia entre name e email respecto al id

# Identificar qué filas de la tabla TIENEN un formato "invalid_"
invalid_rows = customers['email'].str.contains('invalid_')

# Extraer el número del email actual para comparar
num_invalid_email = customers['email'].str.extract(r'invalid_(\d+)')[0]

# Comparar numeros: Solo en las filas que son 'invalid_'
coinciden_num = num_name[invalid_rows] == num_invalid_email[invalid_rows]

invalid_rows.sum(), coinciden_num.sum()

(np.int64(500), np.int64(500))

#### Exporación inconsistencia monthly_value

In [297]:
val_mo = contracts['monthly_value'].describe()
val_mo
#se observa que el valor mínimo es negativo y el máximo es muy alto, lo que sugiere posibles errores en los datos.
# La desviacion estandar es alta, lo que indica una gran variabilidad en los valores mensuales de los contratos.


count    5500.000000
mean       88.058615
std       444.796588
min        -1.000000
25%        42.277500
50%        68.725000
75%        94.470000
max      9999.990000
Name: monthly_value, dtype: float64

In [298]:
valor_menor_0 = contracts[contracts["monthly_value"] < 0]
valor_menor_0

,contract_id,customer_id,plan_type,monthly_value,status,start_date
111,CON_000112,C_04018,M2,-1.0,-1,2021-02-05
112,CON_000113,C_00311,M2,-1.0,1,2020-02-14
113,CON_000114,C_00740,F2,-1.0,1,2020-12-26
114,CON_000115,C_00461,M2,-1.0,1,2021-06-20
115,CON_000116,C_01906,F2,-1.0,1,2020-07-26
116,CON_000117,C_04929,F2,-1.0,0,2020-02-12
117,CON_000118,C_01793,M1,-1.0,1,2020-10-08
118,CON_000119,C_00164,F1,-1.0,1,2022-02-15
119,CON_000120,C_04313,F2,-1.0,1,2020-08-15
120,CON_000121,C_03916,F2,-1.0,1,2021-11-24


In [299]:
valor_mayor_9999 = contracts[contracts["monthly_value"] >= 9999]
valor_mayor_9999

,contract_id,customer_id,plan_type,monthly_value,status,start_date
100,CON_000101,C_00232,F2,9999.99,1,2021-02-11
101,CON_000102,C_02881,F1,9999.99,1,2021-02-09
102,CON_000103,C_04651,F2,9999.99,1,2021-08-04
103,CON_000104,C_04618,M2,9999.99,1,2020-12-22
104,CON_000105,C_01165,M2,9999.99,1,2021-03-24
105,CON_000106,C_04613,M2,9999.99,1,2021-06-20
106,CON_000107,C_01543,F1,9999.99,1,2020-12-22
107,CON_000108,C_04033,F2,9999.99,-1,2021-11-26
108,CON_000109,C_04442,F2,9999.99,0,2023-02-09
109,CON_000110,C_02532,M1,9999.99,1,2021-05-18


In [300]:
# Unir usage con contracts (por contract_id) y luego con customers (por customer_id)    
df_tres_tablas = usage.merge(contracts, on='contract_id').merge(customers, on='customer_id')

In [301]:
# Calcular matriz de correlación
matriz_corr = df_tres_tablas.corr(numeric_only=True)

# Ver la correlación de una columna
print(matriz_corr['monthly_value'])

quantity          0.000264
monthly_value     1.000000
status            0.000185
internal_score   -0.006212
Name: monthly_value, dtype: float64


In [302]:
# Analizar correlación aunque ya veo que no hay mucha relación.
contract_usage = contracts[contracts['contract_id'] == 'CON_000136']
customer_Contracts = customers[customers['customer_id'] == 'C_02990']
usage_Contracts = usage[usage['contract_id'] == 'CON_000136']
contract_usage, customer_Contracts, usage_Contracts

(    contract_id customer_id plan_type  monthly_value  status  start_date
 135  CON_000136     C_02990        F2         105.79       1  2021-01-27,
      customer_id       name         email registration_date category  \
 2989     C_02990  User_2990  invalid_2990        2022-02-16        C   
 
       internal_score  
 2989            62.8  ,
         usage_id contract_id  quantity            timestamp location
 23427  U_0023428  CON_000136      0.56  2024-06-10 19:45:53       DE
 26404  U_0026405  CON_000136      4.81  2024-09-21 16:53:18       DE
 32905  U_0032906  CON_000136      6.47  2024-04-13 03:06:26       ??
 49994  U_0049995  CON_000136      3.31  2024-03-02 10:42:19       UK
 51801  U_0051802  CON_000136      7.01  2024-04-14 13:41:54       DE
 54719  U_0054720  CON_000136      0.01  2024-05-14 05:54:24       UK
 58918  U_0058919  CON_000136      4.38  2024-10-26 08:55:58       UK
 59681  U_0059682  CON_000136      8.04  2024-12-13 01:25:53       ES)

In [303]:

#Explorar .describe() sin outliers
p01 = contracts["monthly_value"].quantile(0.002)
p99 = contracts["monthly_value"].quantile(0.998)
clean = contracts[contracts["monthly_value"].between(p01, p99)]
print(clean["monthly_value"].describe())

count    5479.000000
mean       68.321316
std        30.272708
min        15.030000
25%        42.495000
50%        68.710000
75%        94.345000
max       119.990000
Name: monthly_value, dtype: float64


#### Explorando Status del contrato

In [304]:
contracts.head()

,contract_id,customer_id,plan_type,monthly_value,status,start_date
0,CON_000001,C_00995,F1,51.48,0,2020-07-26
1,CON_000002,C_00098,M2,112.79,1,2021-11-26
2,CON_000003,C_00506,F1,30.34,1,2020-02-23
3,CON_000004,C_01821,F1,81.40,1,2022-07-18
4,CON_000005,C_01071,F1,77.26,1,2022-11-25


In [305]:
# Ver la correlación de una columna (reutilizo)
print(matriz_corr['status'])

quantity          0.002819
monthly_value     0.000185
status            1.000000
internal_score    0.001184
Name: status, dtype: float64


In [306]:
# Hipótesis: que si un status es -1 = cancelado, no hya consumo reciente
# Unir usage con contracts (necesito status y contract_id)
hipotesis_status1 = usage.merge(contracts[['contract_id', 'status']], on='contract_id', how='left')

In [307]:
# Agrupar por status y ver la fecha del último consumo
resultado_hipotesis_status = hipotesis_status1.groupby('status')['timestamp'].agg(['max', 'min', 'count'])
resultado_hipotesis_status


,max,min,count
status,,,
-1.0,2024-12-30 21:49:23,2024-01-01 11:25:29,6335
0.0,2024-12-30 22:23:09,2024-01-01 00:56:08,5773
1.0,2024-12-30 23:48:58,2024-01-01 00:03:42,47391


##### Descarto hipótesis, por el momento tomaré la interpretación más típica ya que no tengo datos a día de hoy y no tengo evidancias que me indiquen lo contrario.

#### Exploro tabla usage columna c_ref

In [308]:
# Que datos de usage tienen coincidencias en contracts en la columna contract_id
contract_id_coinciden = usage['contract_id'].isin(contracts['contract_id'])

# Filas que SÍ tienen coincidencia
usage_con_contrato = usage[contract_id_coinciden]

# Filas que NO tienen coincidencia (el símbolo ~ invierte el filtro)
usage_sin_contrato = usage[~contract_id_coinciden]

print(len(usage_con_contrato), len(usage_sin_contrato))

print(usage_sin_contrato)

59499 501
      usage_id contract_id  quantity            timestamp location
0    U_0000001  CON_999999     24.65  2024-06-01 15:33:49       UK
1    U_0000002  CON_999999      1.81  2024-07-20 01:49:33       UK
2    U_0000003  CON_999999     13.21  2024-08-18 16:51:04       US
3    U_0000004  CON_999999      3.68  2024-08-17 18:10:10       US
4    U_0000005  CON_999999     33.96  2024-10-09 14:13:46       DE
..         ...         ...       ...                  ...      ...
496  U_0000497  CON_999999      3.67  2024-10-12 15:29:11       ES
497  U_0000498  CON_999999     10.63  2024-01-27 16:16:43       UK
498  U_0000499  CON_999999      7.40  2024-07-13 14:56:19       ??
499  U_0000500  CON_999999      8.48  2024-06-21 16:19:18       US
500  U_0000501  CON_999999      1.89  2024-06-30 02:42:40       US

[501 rows x 5 columns]


In [309]:
# cuantos contract_id son igual a CON_999999     
contratos_con_id_999999 = usage[usage['contract_id'] == 'CON_999999']
print(len(contratos_con_id_999999)) 
total_registros = len(usage)
print(total_registros)
contratos_id_999999 = (usage['contract_id'] == 'CON_999999').sum()
print(contratos_id_999999)

501
60000
501


In [310]:
# Saber cuanto me represnta ese contrato en porcentaje sobre el total de registros, para evaluar si eliminarlo o no
porcentaje = (contratos_id_999999 / total_registros) * 100
porcentaje


np.float64(0.835)

#### 

#### Anomalias de cantidad de uso

In [311]:
print(matriz_corr['quantity'])

quantity          1.000000
monthly_value     0.000264
status            0.002819
internal_score    0.007061
Name: quantity, dtype: float64


In [312]:
usage['quantity'].describe()

count    60000.000000
mean         8.195222
std         20.107887
min         -5.000000
25%          2.310000
50%          5.560000
75%         11.080000
max       1440.000000
Name: quantity, dtype: float64

In [313]:
usage.info()

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   usage_id     60000 non-null  str    
 1   contract_id  60000 non-null  str    
 2   quantity     60000 non-null  float64
 3   timestamp    60000 non-null  str    
 4   location     60000 non-null  str    
dtypes: float64(1), str(4)
memory usage: 2.3 MB


In [314]:
neg  = usage[usage["quantity"] < 0]
valor_alto = usage[usage["quantity"] >= 1440]

len(neg), len(valor_alto)

#pocos consumos negativos y pocos consumos grandes, lo que sugiere que podrían ser errores o casos atípicos.

(11, 10)

In [315]:
q_clean = usage[usage["quantity"].between(0.01, 1439)]
q_clean["quantity"].describe()
# se confirma con el .describe() ya que no hay cambios significativos. Creo que el valor negativo 
# si es consumo pueden ser devoluciones y el consumo alto ser atipico pero real

count    59942.000000
mean         7.963837
std          7.910600
min          0.010000
25%          2.320000
50%          5.560000
75%         11.080000
max         83.910000
Name: quantity, dtype: float64

In [316]:
# Sabes si hay relacion con la variable location
valor_alto["location"].value_counts()

location
US    4
DE    3
ES    1
??    1
FR    1
Name: count, dtype: int64

In [317]:
neg["location"].value_counts()

location
US    5
UK    3
ES    1
DE    1
FR    1
Name: count, dtype: int64

# 2. Detección y Resolución de Anomalías Contextuales

##### Según lo explorado anteriormente llego a la conclusión de que la BBDD, en la tabla Customers, tiene un patrón entre las columnas customer_id y name y por lo tanto, al tener datos de email y user diferentes llego a la conclusión que son clientres diferentes y se les a asignado mal su customer_id. Corrijo.

In [318]:
#Solución para inconsistencia de id_cli y full_name

# Extraer el número de la columna 'name' (ej: de 'User_4996' extrae '4996')
numero_name = customers['name'].str.extract(r'(\d+)')[0]

# Convertir a entero y luego a string con formato 'C_' seguido de 5 dígitos (rellenando con ceros)
# El formato :05d asegura que siempre tenga 5 dígitos (04996)
customers['customer_id'] = numero_name.apply(lambda x: f"C_{int(x):05d}")

Solución = customers[customers['customer_id'] == 'C_04996']
customer_id_nunique2 = customers['customer_id'].nunique()
Solución
customer_id_nunique2

5000

##### Como hemos visto hay errores de email con el patrón invalid_(num_name), todos los errores coinciden, se estable patrón correcto que es user_(num_name)@domain.com
##### Se descarta que hayan otros tipos de inconsistencia y que el user de email coincide con el de name, anteriormente vimos que el de name coincide con el de customer.

In [319]:
# Aplicar el cambio solo a las filas detectadas
customers.loc[errores, 'email'] = "user_" + num_name[errores] + "@domain.com"

email_ok=len(customers[customers['email'] != email_esperado])
email_ok, customers[['name', 'email']]

(0,
            name                 email
 0        User_1     user_1@domain.com
 1        User_2     user_2@domain.com
 2        User_3     user_3@domain.com
 3        User_4     user_4@domain.com
 4        User_5     user_5@domain.com
 ...         ...                   ...
 4995  User_4996  user_4996@domain.com
 4996  User_4997  user_4997@domain.com
 4997  User_4998  user_4998@domain.com
 4998  User_4999  user_4999@domain.com
 4999  User_5000  user_5000@domain.com
 
 [5000 rows x 2 columns])

##### Para la interpretación y validación de val_mo, entiendo que es el valor mensual del contrato, cuando veo del desglose del .describe() veo que hay valores atipicos que no tienen sentido, al hacer el describe excluyendo estos valores veo que me da un resultado lógico. Los valores atipicos son 21 filas de un total de 5500 filas, no es un valor muy alto del total. Opto por considerar estos valores como nulos ya que considero que se han introducido mal.

In [320]:
# Definir la condición: valores menores a 0 O mayores/iguales a 9999
condicion = (contracts['monthly_value'] < 0) | (contracts['monthly_value'] >= 9999)

# Aplicar la máscara: si se cumple la condición, pone NaN
contracts['monthly_value'] = contracts['monthly_value'].mask(condicion, np.nan)

# Verificar cuántos nulos se crearon
num_nulos_val= contracts['monthly_value'].isna().sum()
num_nulos_val

np.int64(21)

##### Creo una nueva columna con el status_name, tomo la decisión de usar la interpretación típica ya que se trata de un contrato que puede tener estados, no hay evidencias que sea este relacionado con otra variable, descarto la hipóteisis de que pueda estar relacionado con timestamp de tabla Usage. Confirmo que la mayoria de los contratos son de tipo 1 = activo, lo cual tiene sentido en la mayoria de los casos


In [321]:
# Dicc de mapeo
diccionario_status = {
    1: 'activo',
    0: 'inactivo',
    -1: 'cancelado'
}

# Creo una nueva columna con los nombres de status
contracts['status_name'] = contracts['status'].map(diccionario_status)

# Ver cuántos hay de cada uno
cantidad_contratos= contracts['status_name'].value_counts()

cantidad_contratos, contracts.head()

(status_name
 activo       4386
 cancelado     586
 inactivo      528
 Name: count, dtype: int64,
   contract_id customer_id plan_type  monthly_value  status  start_date  \
 0  CON_000001     C_00995        F1          51.48       0  2020-07-26   
 1  CON_000002     C_00098        M2         112.79       1  2021-11-26   
 2  CON_000003     C_00506        F1          30.34       1  2020-02-23   
 3  CON_000004     C_01821        F1          81.40       1  2022-07-18   
 4  CON_000005     C_01071        F1          77.26       1  2022-11-25   
 
   status_name  
 0    inactivo  
 1      activo  
 2      activo  
 3      activo  
 4      activo  )

##### El unico datos que no tiene coincidencia con la tabla de contracts es el CON_999999, veo que son 501 registros. Por lo tanto tengo dos opciones, elimino en uso de este contrato, los 501 registros (menos del 1% del total) o creo un registro de este contrato, evaluo que los datos que me genera la tabla usage no son suficientes para tener un registro completo del contrato, sin tipo de plan, valor mensual, status y fecha de inicio por lo tanto mi elección es eliminar los registros de usage.

In [322]:
# Elimino todos los registros de usage que su contract_id es igual a CON_999999
usage = usage[usage['contract_id'] != 'CON_999999']
print(len(usage))

59499


##### He tomado la decision de eliminar valores negetivos ya que pueden significar devolución pero como no estoy segura que eso sea posible optaré por eliminarlos ya que es consumo negativo, pero los valores mas altos los dejaré ya que son pocos y es tipico en consumo que exista la posibilidad de alguna vez consumir mucho. No es algo común pero puede pasar y la empresa debe considerar que pueden aparecer outliers que no son errores.

In [324]:
usage = usage[usage['quantity'] != -5]
print(len(usage))

59488


# 3. Detección de Inconsistencias Lógicas y de Negocio Cruzadas 